In [20]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent

RAW_DATA = BASE_DIR / "data" / "raw"
PROCESSED_DATA = BASE_DIR / "data" / "processed"
DB_DIR = BASE_DIR / "data" / "db"

In [2]:
# nav_history cleaning

nav_history = pd.read_csv(RAW_DATA / "02_nav_history.csv")
nav_history.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [3]:
nav_history["date"]=pd.to_datetime(nav_history["date"])
nav_history.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [4]:
nav_history=nav_history.sort_values(["amfi_code", "date"])
nav_history.head()

,amfi_code,date,nav
5750,100016,2022-01-03,520.4608
5751,100016,2022-01-04,515.0971
5752,100016,2022-01-05,521.7239
5753,100016,2022-01-06,515.7880
5754,100016,2022-01-07,515.1639


In [5]:
print(nav_history.shape)
nav_history=nav_history.drop_duplicates()
print(nav_history.shape)

(46000, 3)
(46000, 3)


In [6]:
nav_history["nav"] = (
    nav_history.groupby("amfi_code")["nav"]
    .ffill()
)

In [7]:
invalid_nav = nav_history[nav_history["nav"] <= 0]
print(len(invalid_nav))

0


In [8]:
nav_history.to_csv(PROCESSED_DATA / "nav_history_cleaned.csv", index=False)

In [9]:
# investor_transactions cleaning

investor_transactions = pd.read_csv(RAW_DATA / "08_investor_transactions.csv")
investor_transactions.head()
investor_transactions["transaction_type"].unique()

array(['SIP', 'Redemption', 'Lumpsum'], dtype=object)

In [10]:
invalid_investor_transactions=investor_transactions[investor_transactions["amount_inr"]<=0]
print(len(invalid_investor_transactions))

0


In [11]:
investor_transactions["transaction_date"]=pd.to_datetime(investor_transactions["transaction_date"])

In [12]:
print(investor_transactions["kyc_status"].unique())
investor_transactions["kyc_status"].value_counts()


['Verified' 'Pending']


kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64

In [13]:
investor_transactions.to_csv(
    PROCESSED_DATA / "investor_transactions_cleaned.csv",
    index=False
)

In [14]:
# scheme_performance cleaning
scheme_performance = pd.read_csv(RAW_DATA / "07_scheme_performance.csv")
scheme_performance.head()

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [15]:
scheme_performance.dtypes

amfi_code               int64
scheme_name            object
fund_house             object
category               object
plan                   object
return_1yr_pct        float64
return_3yr_pct        float64
return_5yr_pct        float64
benchmark_3yr_pct     float64
alpha                 float64
beta                  float64
sharpe_ratio          float64
sortino_ratio         float64
std_dev_ann_pct       float64
max_drawdown_pct      float64
aum_crore               int64
expense_ratio_pct     float64
morningstar_rating      int64
risk_grade             object
dtype: object

In [16]:
scheme_performance["return_anomaly"] = (
    (scheme_performance["return_1yr_pct"] > 100)
    | (scheme_performance["return_1yr_pct"] < -100)
    | (scheme_performance["return_3yr_pct"] > 100)
    | (scheme_performance["return_3yr_pct"] < -100)
    | (scheme_performance["return_5yr_pct"] > 100)
    | (scheme_performance["return_5yr_pct"] < -100)
)

scheme_performance["expense_ratio_valid"] = (
    (scheme_performance["expense_ratio_pct"] >= 0.1)
    &
    (scheme_performance["expense_ratio_pct"] <= 2.5)
)
print("Return anomalies:",
      scheme_performance["return_anomaly"].sum())


print("Invalid expense ratios:",
      (~scheme_performance["expense_ratio_valid"]).sum())


Return anomalies: 0
Invalid expense ratios: 0


In [17]:
scheme_performance.drop(
    columns=["expense_ratio_valid"],
    inplace=True
)
scheme_performance.drop(
    columns=["return_anomaly"],
    inplace=True
)
scheme_performance.to_csv(
    PROCESSED_DATA / "scheme_performance_cleaned.csv",
    index=False
)

In [21]:
from sqlalchemy import create_engine
DB_DIR.mkdir(parents=True, exist_ok=True)

# Create SQLite database
engine = create_engine(
    f"sqlite:///{DB_DIR / 'bluestock_mf.db'}"
)


In [22]:
# Load cleaned datasets
fund_master = pd.read_csv(RAW_DATA / "01_fund_master.csv")

nav_history = pd.read_csv(PROCESSED_DATA / "nav_history_cleaned.csv")

investor_transactions = pd.read_csv(PROCESSED_DATA / "investor_transactions_cleaned.csv")

scheme_performance = pd.read_csv(PROCESSED_DATA / "scheme_performance_cleaned.csv")

aum = pd.read_csv(RAW_DATA / "03_aum_by_fund_house.csv")

In [23]:
# Load into SQLite
fund_master.to_sql(
    "dim_fund",
    engine,
    if_exists="replace",
    index=False
)

nav_history.to_sql(
    "fact_nav",
    engine,
    if_exists="replace",
    index=False
)

investor_transactions.to_sql(
    "fact_transactions",
    engine,
    if_exists="replace",
    index=False
)

scheme_performance.to_sql(
    "fact_performance",
    engine,
    if_exists="replace",
    index=False
)

aum.to_sql(
    "fact_aum",
    engine,
    if_exists="replace",
    index=False
)

print("Database loaded successfully")

Database loaded successfully


In [24]:
for table in [
    "dim_fund",
    "fact_nav",
    "fact_transactions",
    "fact_performance",
    "fact_aum"
]:
    print(table)
    print(
        pd.read_sql(
            f"SELECT COUNT(*) AS rows FROM {table}",
            engine
        )
    )

dim_fund
   rows
0    40
fact_nav
    rows
0  46000
fact_transactions
    rows
0  32778
fact_performance
   rows
0    40
fact_aum
   rows
0    90


In [25]:
query = """
SELECT
state,
COUNT(*) AS transactions
FROM fact_transactions
GROUP BY state
ORDER BY COUNT(*) DESC;
"""

pd.read_sql(query, engine)

,state,transactions
0,Punjab,2965
1,Madhya Pradesh,2931
2,Tamil Nadu,2806
3,Gujarat,2780
4,West Bengal,2748
5,Haryana,2736
6,Telangana,2718
7,Uttar Pradesh,2695
8,Delhi,2677
9,Karnataka,2621
